# Araz — Preprocessing: Daily Rental & Text Features

## Objective

This notebook is responsible for preprocessing the **Daily Rental Features** and **Text Features** of the Divar real-estate dataset.

The preprocessing workflow is divided into the following stages:

1. Load only the relevant columns.
2. Inspect data types and distributions.
3. Analyze missing values and determine their meaning.
4. Detect invalid or inconsistent values.
5. Clean text features.
6. Clean daily-rental numerical features.
7. Document every preprocessing decision before applying it to the full dataset.

> **Important:** Missing values will not be filled or removed blindly.  
> First, their meaning will be investigated because some missing values may indicate that a feature is not applicable to a specific type of advertisement.

## 1. Setup

First, the required libraries are imported.

- **Pandas** is used for loading, inspecting, and preprocessing tabular data.
- **NumPy** is used for numerical operations and handling missing values.
- **Pathlib** is used to manage file paths in a platform-independent way.

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

In [3]:
Path.cwd()

WindowsPath('c:/Users/HP/Desktop/Projects/Learn/ML&AI/project1/notebooks')

## 2. Dataset Path

The raw dataset is stored in the `data/raw` directory.

Before loading the dataset, we first verify that the file path is correct. This prevents file-loading errors and keeps the project structure organized.

In [4]:
DATA_PATH = Path("../data/raw/Divar.csv")
DATA_PATH.exists()

True

## 3. Inspecting Dataset Columns

Before loading the full dataset, only the column names are read.

Since the dataset is large, reading just the header allows us to verify the available features without unnecessary memory usage.

In [5]:
all_columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()

print("Number of columns:", len(all_columns))
all_columns

Number of columns: 61


['Unnamed: 0',
 'cat2_slug',
 'cat3_slug',
 'city_slug',
 'neighborhood_slug',
 'created_at_month',
 'user_type',
 'description',
 'title',
 'rent_mode',
 'rent_value',
 'rent_to_single',
 'rent_type',
 'price_mode',
 'price_value',
 'credit_mode',
 'credit_value',
 'rent_credit_transform',
 'transformable_price',
 'transformable_credit',
 'transformed_credit',
 'transformable_rent',
 'transformed_rent',
 'land_size',
 'building_size',
 'deed_type',
 'has_business_deed',
 'floor',
 'rooms_count',
 'total_floors_count',
 'unit_per_floor',
 'has_balcony',
 'has_elevator',
 'has_warehouse',
 'has_parking',
 'construction_year',
 'is_rebuilt',
 'has_water',
 'has_warm_water_provider',
 'has_electricity',
 'has_gas',
 'has_heating_system',
 'has_cooling_system',
 'has_restroom',
 'has_security_guard',
 'has_barbecue',
 'building_direction',
 'has_pool',
 'has_jacuzzi',
 'has_sauna',
 'floor_material',
 'property_type',
 'regular_person_capacity',
 'extra_person_capacity',
 'cost_per_extra_p

## 4. Selecting Araz's Features

This notebook focuses on two groups of features:

### Text Features
- `title`
- `description`

### Daily Rental Features
- `regular_person_capacity`
- `extra_person_capacity`
- `cost_per_extra_person`
- `rent_price_on_regular_days`
- `rent_price_on_special_days`
- `rent_price_at_weekends`

Before loading the data, the existence of all required columns is checked to avoid errors caused by missing or incorrectly named features.

In [6]:
araz_columns = [
    "title",
    "description",
    "regular_person_capacity",
    "extra_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

In [7]:
missing_columns = [
    col for col in araz_columns
    if col not in all_columns
]

missing_columns

[]

### Context Features

Some additional columns are temporarily included to provide context for the daily-rental features.

These columns are not part of the preprocessing responsibility of this notebook. They are only used to understand whether missing values in rental-related features represent actual missing data or whether those features are not applicable to certain advertisement types.

In [8]:
candidate_helper_columns = [
    "cat2_slug",
    "cat3_slug",
    "rent_mode",
    "rent_type"
]

helper_columns = [
    col for col in candidate_helper_columns
    if col in all_columns
]

helper_columns

['cat2_slug', 'cat3_slug', 'rent_mode', 'rent_type']

## 5. Loading the Required Features

The complete dataset is loaded for the selected features.

Instead of loading every column from the original CSV file, only the features required for this preprocessing task and a small number of contextual features are loaded.

This approach preserves all observations while reducing unnecessary memory consumption.

In [9]:
selected_columns = araz_columns + helper_columns

df = pd.read_csv(
    DATA_PATH,
    usecols=selected_columns,
    low_memory=False
)

df.shape

(1000000, 12)

### Loading the Full Dataset

The complete dataset is loaded using only the selected features.

`low_memory=False` is used because Pandas may otherwise infer different data types for different chunks of a large CSV file. Reading the file with this option provides more consistent dtype inference across the selected columns.

In [10]:
print("Shape:", df.shape)

memory_gb = df.memory_usage(deep=True).sum() / (1024 ** 3)

print(f"Memory usage: {memory_gb:.2f} GB")

Shape: (1000000, 12)
Memory usage: 0.63 GB


## 6. Initial Data Inspection

Before applying any preprocessing steps, the selected features are inspected to understand:

- their data types,
- the number of non-null values,
- the amount of missing data,
- and whether any columns have unexpected representations.

No values are modified at this stage.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 12 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   cat2_slug                   1000000 non-null  str    
 1   cat3_slug                   999999 non-null   str    
 2   description                 1000000 non-null  str    
 3   title                       999946 non-null   str    
 4   rent_mode                   352994 non-null   str    
 5   rent_type                   103961 non-null   str    
 6   regular_person_capacity     29870 non-null    float64
 7   extra_person_capacity       24009 non-null    str    
 8   cost_per_extra_person       10241 non-null    float64
 9   rent_price_on_regular_days  18068 non-null    float64
 10  rent_price_on_special_days  10463 non-null    float64
 11  rent_price_at_weekends      13551 non-null    float64
dtypes: float64(5), str(7)
memory usage: 642.2 MB


In [12]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

missing_report = missing_report.sort_values(
    "missing_percent",
    ascending=False
)

missing_report

,missing_count,missing_percent
cost_per_extra_person,989759,98.9759
rent_price_on_special_days,989537,98.9537
rent_price_at_weekends,986449,98.6449
rent_price_on_regular_days,981932,98.1932
extra_person_capacity,975991,97.5991
regular_person_capacity,970130,97.0130
rent_type,896039,89.6039
rent_mode,647006,64.7006
title,54,0.0054
cat3_slug,1,0.0001


In [13]:
df.isna().mean()

cat2_slug                     0.000000
cat3_slug                     0.000001
description                   0.000000
title                         0.000054
rent_mode                     0.647006
rent_type                     0.896039
regular_person_capacity       0.970130
extra_person_capacity         0.975991
cost_per_extra_person         0.989759
rent_price_on_regular_days    0.981932
rent_price_on_special_days    0.989537
rent_price_at_weekends        0.986449
dtype: float64

## 7. Investigating Rental Context

The daily-rental features contain a very high percentage of missing values.

However, these missing values may not represent missing information. They may indicate that the corresponding advertisement is not a daily-rental listing.

Therefore, before applying any imputation or deletion strategy, the contextual rental features `rent_mode` and `rent_type` are inspected.

In [14]:
df["rent_mode"].value_counts(dropna=False)

rent_mode
NaN       647006
مقطوع     292081
مجانی      59241
توافقی      1672
Name: count, dtype: int64

In [15]:
df["rent_type"].value_counts(dropna=False)

rent_type
NaN            896039
rent_credit    103814
full_credit       147
Name: count, dtype: int64

## 8. Identifying Daily Rental Listings

The daily-rental features are populated for only a small portion of the dataset.

Before treating their missing values, we need to determine which types of advertisements actually use these features.

A row is temporarily considered related to daily-rental information if at least one of the daily-rental features contains a value.

In [16]:
daily_columns = [
    "regular_person_capacity",
    "extra_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

daily_mask = df[daily_columns].notna().any(axis=1)

In [17]:
daily_count = daily_mask.sum()
daily_percent = daily_mask.mean() * 100

print("Rows with at least one daily-rental feature:", daily_count)
print(f"Percentage of dataset: {daily_percent:.2f}%")

Rows with at least one daily-rental feature: 29870
Percentage of dataset: 2.99%


In [18]:
df.loc[daily_mask, "cat3_slug"].value_counts().head(20)

cat3_slug
suite-apartment    16457
villa              12878
workspace            535
Name: count, dtype: int64

## 9. Missing Values Within Daily-Rental Listings

The previous analysis showed that daily-rental features are only applicable to a small subset of the dataset.

Therefore, the high missing percentages observed in the full dataset are mostly structural and should not automatically be treated as missing-data problems.

The next step is to analyze missing values **only within rows that contain daily-rental information**. This helps distinguish:

- values that are not applicable to non-daily listings,
- from values that are genuinely missing within daily-rental listings.

In [19]:
daily_df = df.loc[daily_mask, daily_columns].copy()

daily_df.shape

(29870, 6)

In [20]:
daily_missing_report = pd.DataFrame({
    "missing_count": daily_df.isna().sum(),
    "missing_percent": daily_df.isna().mean() * 100
})

daily_missing_report.sort_values(
    "missing_percent",
    ascending=False
)

,missing_count,missing_percent
cost_per_extra_person,19629,65.714764
rent_price_on_special_days,19407,64.971543
rent_price_at_weekends,16319,54.633411
rent_price_on_regular_days,11802,39.511215
extra_person_capacity,5861,19.621694
regular_person_capacity,0,0.000000


## 10. Inspecting Extra Person Capacity

`extra_person_capacity` is expected to represent a numerical capacity, but Pandas inferred it as a string column.

Before converting its data type, the unique values are inspected to determine whether the column contains:

- numerical values stored as text,
- special categorical values,
- inconsistent representations,
- or invalid entries.

No conversion is applied before understanding the existing values.

In [21]:
df.loc[daily_mask, "extra_person_capacity"].value_counts(
    dropna=False
).head(30)

extra_person_capacity
2      6849
NaN    5861
1      3910
4      2843
3      2086
5      2062
6      1165
10     1066
0       923
30+     724
8       617
7       429
20      304
15      262
12      208
9       104
25       78
14       61
11       52
16       41
29       38
24       32
13       31
18       23
23       14
22       14
17       13
28       12
26       12
19        9
Name: count, dtype: int64

In [22]:
df.loc[daily_mask, "extra_person_capacity"].unique()[:50]

<ArrowStringArray>
[  '6',   '7',   '2',   '1',   '0',   '4',   '3',   nan,  '10',   '8',  '15',
  '25',   '5', '30+',  '29',  '11',  '12',  '20',  '14',  '28',  '18',  '13',
   '9',  '19',  '17',  '24',  '27',  '21',  '23',  '16',  '26',  '22', '4.0',
 '5.0', '1.0', '2.0', '6.0']
Length: 37, dtype: str

### Investigating Non-Numeric Capacity Values

Although `extra_person_capacity` conceptually represents a numerical quantity, it is stored as a string column.

Most values appear to be numeric strings, but special values such as `30+` are also present.

Before converting the column to a numeric data type, we identify all values that cannot be parsed as numbers. This prevents losing information during conversion.

In [23]:
extra_raw = (
    df.loc[daily_mask, "extra_person_capacity"]
    .astype("string")
    .str.strip()
)

extra_numeric_test = pd.to_numeric(
    extra_raw,
    errors="coerce"
)

non_numeric_mask = (
    extra_raw.notna()
    & extra_numeric_test.isna()
)

extra_raw[non_numeric_mask].value_counts()

extra_person_capacity
30+    724
Name: count, dtype: int64[pyarrow]

### Capacity Distribution Check

Only one non-numeric value was found in `extra_person_capacity`: `30+`.

Before deciding how to encode this value, the numerical distribution of the column is inspected.

This helps determine whether `30+` represents a capped upper category and whether replacing it with a numerical value such as `30` would be reasonable.

In [24]:
extra_capacity_temp = pd.to_numeric(
    extra_raw.replace("30+", "30"),
    errors="coerce"
)

extra_capacity_temp.describe()

count     24009.0
mean     4.794119
std      5.997498
min           0.0
25%           2.0
50%           3.0
75%           5.0
max          30.0
Name: extra_person_capacity, dtype: Float64

In [25]:
extra_capacity_temp.value_counts().sort_index().tail(15)

extra_person_capacity
16.0     41
17.0     13
18.0     23
19.0      9
20.0    304
21.0      6
22.0     14
23.0     14
24.0     32
25.0     78
26.0     12
27.0      9
28.0     12
29.0     38
30.0    724
Name: count, dtype: Int64

### Checking the Upper Capacity Boundary

Before converting `30+` to a numerical value, we verify whether the exact value `30` also exists in the original data.

This is important because `30` and `30+` do not have exactly the same meaning.  
Merging them without checking could result in loss of information.


In [26]:
extra_raw[
    extra_raw.isin(["28", "29", "30", "30+"])
].value_counts()

extra_person_capacity
30+    724
29      38
28      12
Name: count, dtype: int64[pyarrow]

In [27]:
df.loc[
    df["extra_person_capacity"].eq("30+"),
    [
        "cat3_slug",
        "regular_person_capacity",
        "extra_person_capacity",
        "cost_per_extra_person"
    ]
].head(10)

,cat3_slug,regular_person_capacity,extra_person_capacity,cost_per_extra_person
4643,villa,6.0,30+,NaN
5812,villa,50.0,30+,NaN
7004,villa,20.0,30+,50000.0
7600,villa,4.0,30+,400000.0
8809,villa,10.0,30+,NaN
9211,suite-apartment,1.0,30+,50000.0
9249,villa,8.0,30+,200000.0
9661,villa,8.0,30+,50000.0
9806,villa,50.0,30+,NaN
10006,villa,50.0,30+,NaN


### Cleaning `extra_person_capacity`

The inspection showed that `30+` is the only non-numeric value in this feature and the exact value `30` is not stored as a separate raw category.

Therefore, `30+` is encoded as `30` as a conservative capped representation. All other numeric strings are converted to numbers and the feature is stored as Pandas nullable `Int64`, preserving missing values.


In [ ]:
df["extra_person_capacity"] = (
    df["extra_person_capacity"]
    .astype("string")
    .str.strip()
    .replace("30+", "30")
    .pipe(pd.to_numeric, errors="coerce")
    .astype("Int64")
)

print("dtype:", df["extra_person_capacity"].dtype)
print("missing:", df["extra_person_capacity"].isna().sum())


## 12. Profiling Daily Rental Numerical Features

The remaining daily-rental numerical features are inspected together to identify:

- invalid negative values,
- zero values,
- fractional values,
- extreme values,
- and overall distribution patterns.

This allows the preprocessing strategy to be defined consistently across all daily-rental features.

In [28]:
numeric_daily_cols = [
    "regular_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

profile = []

for col in numeric_daily_cols:
    s = df.loc[daily_mask, col]

    profile.append({
        "column": col,
        "count": s.notna().sum(),
        "missing": s.isna().sum(),
        "min": s.min(),
        "median": s.median(),
        "mean": s.mean(),
        "max": s.max(),
        "negative_count": (s < 0).sum(),
        "zero_count": (s == 0).sum(),
        "fractional_count": (
            s.notna() & (s % 1 != 0)
        ).sum()
    })

profile_df = pd.DataFrame(profile)

profile_df

,column,count,missing,min,median,mean,max,negative_count,zero_count,fractional_count
0,regular_person_capacity,29870,0,1.0,4.0,6.557650e+00,5.000000e+01,0,0,0
1,cost_per_extra_person,10241,19629,0.0,100000.0,1.209785e+10,1.111111e+14,0,473,0
2,rent_price_on_regular_days,18068,11802,0.0,800000.0,1.389016e+11,5.006007e+14,0,207,0
3,rent_price_on_special_days,10463,19407,0.0,1200000.0,2.355548e+10,1.111111e+14,0,241,0
4,rent_price_at_weekends,13551,16319,0.0,1100000.0,3.156551e+10,2.002503e+14,0,228,0


In [29]:
df.loc[
    daily_mask,
    numeric_daily_cols
].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
regular_person_capacity,29870.0,6.557650e+00,7.698655e+00,1.0,1.0,2.0,3.0,4.0,7.0,17.0,50.0,5.000000e+01
cost_per_extra_person,10241.0,1.209785e+10,1.103482e+12,0.0,0.0,1.0,50000.0,100000.0,200000.0,500000.0,2500000.0,1.111111e+14
rent_price_on_regular_days,18068.0,1.389016e+11,7.042335e+12,0.0,0.0,400.0,400000.0,800000.0,1600000.0,3500000.0,25000000.0,5.006007e+14
rent_price_on_special_days,10463.0,2.355548e+10,1.542049e+12,0.0,0.0,400.0,600000.0,1200000.0,2500000.0,6000000.0,20000000.0,1.111111e+14
rent_price_at_weekends,13551.0,3.156551e+10,2.434942e+12,0.0,0.0,500.0,550000.0,1100000.0,2500000.0,5000000.0,15000000.0,2.002503e+14


In [30]:
df.loc[
    daily_mask,
    "regular_person_capacity"
].value_counts().sort_index().tail(20)

regular_person_capacity
30.0    152
31.0      2
32.0      3
33.0      6
34.0      5
35.0     31
36.0      1
37.0      5
38.0      3
39.0      1
40.0     90
42.0      4
43.0      1
44.0      3
45.0     17
46.0      3
47.0      1
48.0      1
49.0      4
50.0    548
Name: count, dtype: int64

### Cleaning `regular_person_capacity`

The feature contains only positive integer values between 1 and 50.

No negative, zero, or fractional values were found. Values near the upper boundary also occur repeatedly, so they are not treated as errors based only on their magnitude.

The feature is therefore converted to a nullable integer type without removing any observations.

In [31]:
df["regular_person_capacity"] = (
    df["regular_person_capacity"]
    .astype("Int64")
)

## 13. Investigating Suspicious Daily Rental Prices

The daily-rental price features are highly right-skewed and contain extremely large values compared with their typical ranges.

For example, the maximum values are several orders of magnitude larger than the 99th percentile.

Very small values, including zero, are also present.

Before removing or replacing any values, both tails of the distributions are inspected to distinguish plausible prices from likely data-entry errors or placeholder values.

In [32]:
price_cols = [
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

for col in price_cols:
    s = df.loc[daily_mask, col].dropna()

    print(f"\n{'='*60}")
    print(col)

    print("\nSmallest values:")
    print(s.value_counts().sort_index().head(15))

    print("\nLargest values:")
    print(s.sort_values(ascending=False).head(15).to_list())


cost_per_extra_person

Smallest values:
cost_per_extra_person
0.0     473
1.0      41
2.0       3
3.0       4
4.0       4
5.0       6
6.0       2
8.0       1
10.0     14
11.0      8
18.0      1
20.0      7
22.0      2
25.0      1
30.0     15
Name: count, dtype: int64

Largest values:
[111111111111111.0, 11111111111111.0, 1111111111111.0, 200300400500.0, 200300400500.0, 111111111111.0, 11111111111.0, 11111111111.0, 11111111111.0, 1111111111.0, 1111111111.0, 1111111111.0, 1111111111.0, 1111111111.0, 1111111111.0]

rent_price_on_regular_days

Smallest values:
rent_price_on_regular_days
0.0     207
1.0      48
2.0      10
3.0      12
4.0       2
5.0      10
6.0       3
7.0       2
10.0      3
11.0      5
25.0      1
30.0      1
35.0      1
40.0      1
50.0      4
Name: count, dtype: int64

Largest values:
[500600700800900.0, 500600700800900.0, 400500600800900.0, 400500600700800.0, 111111111111111.0, 111111111111111.0, 111111111111111.0, 111111111111111.0, 111111111111111.0, 11111111111111

In [33]:
suspicious_report = []

for col in price_cols:
    s = df.loc[daily_mask, col]

    suspicious_report.append({
        "column": col,
        "zero": (s == 0).sum(),
        "<= 1,000": (s <= 1_000).sum(),
        "<= 10,000": (s <= 10_000).sum(),
        "> 10,000,000": (s > 10_000_000).sum(),
        "> 100,000,000": (s > 100_000_000).sum(),
        "> 1,000,000,000": (s > 1_000_000_000).sum(),
    })

pd.DataFrame(suspicious_report)

,column,zero,"<= 1,000","<= 10,000","> 10,000,000","> 100,000,000","> 1,000,000,000"
0,cost_per_extra_person,473,2171,2259,63,37,18
1,rent_price_on_regular_days,207,2024,2219,246,152,61
2,rent_price_on_special_days,241,1042,1208,204,65,25
3,rent_price_at_weekends,228,1311,1488,202,85,35


## 14. Cleaning Daily Rental Price Features

The daily-rental price features contain suspicious values at both ends of their distributions.

Extremely large values are several orders of magnitude larger than the 99th percentile and include clearly irregular patterns such as repeated digits. These values are treated as invalid observations.

Very small values are also suspicious; however, their original meaning is uncertain. They may represent data-entry conventions or unit inconsistencies. Therefore, they are flagged but not automatically modified.

Missing values are preserved because they may represent optional or non-applicable information rather than true missing data.

In [34]:
price_cols = [
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends"
]

EXTREME_PRICE_THRESHOLD = 1_000_000_000

for col in price_cols:
    invalid_high = df[col] > EXTREME_PRICE_THRESHOLD
    df.loc[invalid_high, col] = np.nan

In [35]:
for col in price_cols:
    df[f"{col}_suspicious_low"] = (
        df[col].notna() &
        (df[col] <= 10_000)
    )

In [36]:
df["regular_person_capacity"] = (
    df["regular_person_capacity"]
    .astype("Int64")
)

In [37]:
df[
    [
        "regular_person_capacity",
        "extra_person_capacity",
        *price_cols
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
regular_person_capacity,29870.0,6.55765,7.698655,1.0,3.0,4.0,7.0,50.0
cost_per_extra_person,10223.0,561116.775897,10294388.860894,0.0,50000.0,100000.0,200000.0,666666666.0
rent_price_on_regular_days,18007.0,3209174.431166,31449582.427117,0.0,400000.0,800000.0,1600000.0,1000000000.0
rent_price_on_special_days,10438.0,2870166.49406,19929289.734523,0.0,590000.0,1200000.0,2500000.0,800900999.0
rent_price_at_weekends,13516.0,2945359.690811,24139232.355039,0.0,550000.0,1100000.0,2500000.0,999999999.0


In [38]:
df[price_cols].max()

cost_per_extra_person         6.666667e+08
rent_price_on_regular_days    1.000000e+09
rent_price_on_special_days    8.009010e+08
rent_price_at_weekends        1.000000e+09
dtype: float64

# Text Features Preprocessing

The dataset contains two main text features:

- `title`
- `description`

Text preprocessing will focus on basic normalization and data-quality checks while preserving useful semantic information.

Aggressive operations such as removing numbers, punctuation, or words are avoided at this stage because they may remove meaningful information.

Special attention is also given to potential target leakage, since real-estate advertisements may explicitly mention prices inside their text.

In [39]:
text_cols = ["title", "description"]

text_report = pd.DataFrame({
    "missing": df[text_cols].isna().sum(),
    "unique": df[text_cols].nunique(),
    "empty_string": [
        df[col].fillna("").str.strip().eq("").sum()
        for col in text_cols
    ]
})

text_report

,missing,unique,empty_string
title,54,904092,54
description,0,989449,1


In [40]:
for col in text_cols:
    lengths = df[col].fillna("").str.len()

    print(f"\n{col}")
    print(lengths.describe(
        percentiles=[0.01, 0.5, 0.95, 0.99]
    ))


title
count    1000000.000000
mean          30.552262
std           10.907883
min            0.000000
1%             7.000000
50%           30.000000
95%           48.000000
99%           50.000000
max           59.000000
Name: title, dtype: float64

description
count    1000000.000000
mean         273.619646
std          208.631873
min            1.000000
1%            20.000000
50%          214.000000
95%          718.000000
99%          943.000000
max         1000.000000
Name: description, dtype: float64


## 15. Cleaning Text Features

The `title` and `description` columns are almost complete and contain highly diverse textual information.

The preprocessing strategy is intentionally conservative. The goal is to normalize formatting inconsistencies without removing potentially useful semantic information.

The following operations are applied:

- leading and trailing whitespace is removed,
- repeated whitespace is normalized,
- Persian/Arabic character variants are standardized,
- empty strings are converted to missing values.

Numbers and punctuation are preserved because they may contain useful information.

However, text features must later be checked for potential **target leakage**, since advertisements may explicitly mention prices in their title or description.

In [41]:
text_cols = ["title", "description"]

for col in text_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace("ي", "ی", regex=False)
        .str.replace("ك", "ک", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

In [42]:
text_validation = pd.DataFrame({
    "missing": df[text_cols].isna().sum(),
    "unique": df[text_cols].nunique(),
    "empty_string": [
        df[col].fillna("").str.strip().eq("").sum()
        for col in text_cols
    ]
})

text_validation

,missing,unique,empty_string
title,54,902758,54
description,1,988819,1


In [43]:
for col in text_cols:
    print(
        col,
        "duplicated values:",
        df[col].duplicated().sum()
    )

title duplicated values: 97241
description duplicated values: 11180


In [44]:
df["title_length"] = df["title"].str.len()
df["description_length"] = df["description"].str.len()

df["title_word_count"] = df["title"].str.split().str.len()
df["description_word_count"] = df["description"].str.split().str.len()

### Text Length Features

Additional lightweight features are created from the text columns:

- title length,
- description length,
- title word count,
- description word count.

These features preserve simple information about advertisement verbosity without requiring a full NLP pipeline.

### Potential Target Leakage in Text

Because the final machine-learning task includes price prediction, textual features may contain explicit price information.

Using such information directly could introduce target leakage. Therefore, potential price-related patterns are inspected before deciding whether raw text should be used in the regression model.

In [ ]:
price_keywords = r"قیمت|تومان|میلیون|میلیارد"

for col in text_cols:
    count = (
        df[col]
        .fillna("")
        .str.contains(price_keywords, regex=True)
        .sum()
    )

    print(f"{col}: {count}")


# 16. Final Validation and Integration Notes

Before handing this preprocessing work to the team, the final schema and key quality checks are reviewed.

Important decisions:

- Structural missing values outside daily-rental listings are preserved.
- Missing optional daily-rental values are not blindly imputed.
- `regular_person_capacity` and `extra_person_capacity` are represented as nullable integers.
- Clearly extreme daily-rental price values are replaced with missing values using the documented conservative rule.
- Suspiciously small price values are flagged rather than guessed or rescaled.
- Persian/Arabic character variants and whitespace are normalized in text.
- Lightweight text-length features are created.
- Raw text should **not automatically be used in the final price-regression model** before the team evaluates target-leakage risk.


In [ ]:
final_columns = [
    "title",
    "description",
    "title_length",
    "description_length",
    "title_word_count",
    "description_word_count",
    "regular_person_capacity",
    "extra_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends",
]

print("Final selected dtypes:")
display(df[final_columns].dtypes.to_frame("dtype"))

print("\nFinal missing percentages:")
display(
    (df[final_columns].isna().mean() * 100)
    .sort_values(ascending=False)
    .to_frame("missing_percent")
)

assert str(df["regular_person_capacity"].dtype) == "Int64"
assert str(df["extra_person_capacity"].dtype) == "Int64"
assert df["title"].fillna("").str.strip().eq("").sum() == df["title"].isna().sum()
assert df["description"].fillna("").str.strip().eq("").sum() == df["description"].isna().sum()

print("\nFinal validation passed.")


## Team Integration

The notebook above documents the analysis and reasoning. For merging into the team's main preprocessing pipeline, the same transformations are provided below as a reusable function.

The raw dataset itself should **not** be committed to GitHub.


In [ ]:
def preprocess_araz_features(data):
    out = data.copy()

    # Capacity features
    out["regular_person_capacity"] = pd.to_numeric(
        out["regular_person_capacity"], errors="coerce"
    ).astype("Int64")

    out["extra_person_capacity"] = (
        out["extra_person_capacity"]
        .astype("string")
        .str.strip()
        .replace("30+", "30")
        .pipe(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )

    # Daily-rental prices
    price_cols = [
        "cost_per_extra_person",
        "rent_price_on_regular_days",
        "rent_price_on_special_days",
        "rent_price_at_weekends",
    ]

    extreme_price_threshold = 1_000_000_000

    for col in price_cols:
        out.loc[out[col] > extreme_price_threshold, col] = np.nan
        out[f"{col}_suspicious_low"] = out[col].notna() & (out[col] <= 10_000)

    # Text normalization
    text_cols = ["title", "description"]

    for col in text_cols:
        out[col] = (
            out[col]
            .astype("string")
            .str.replace("ي", "ی", regex=False)
            .str.replace("ك", "ک", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .replace("", pd.NA)
        )

    # Lightweight text features
    out["title_length"] = out["title"].str.len()
    out["description_length"] = out["description"].str.len()
    out["title_word_count"] = out["title"].str.split().str.len()
    out["description_word_count"] = out["description"].str.split().str.len()

    return out
